#Proyecto final – Etapa de transferencia

In [1]:
import pandas as pd
import numpy as np

# Cargar el dataset y realizar la limpieza básica
df = pd.read_csv('/content/kc_house_data.csv')
df = df.drop(columns=["id"])
df["date"] = pd.to_datetime(df["date"])
df = df.drop_duplicates()
df = df[df["bedrooms"] < 15]

print(df.columns.tolist())

['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15']


##Modelo mejorado — agregar variables relevantes

Vamos a incorporar variables de ubicación y calidad, que identificamos como la explicación más probable del R² moderado del modelo base.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Modelo BASE (referencia, ya lo tienes)
X_base = df[["sqft_living", "bathrooms", "yr_built"]]
y = df["price"]

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_base, y, test_size=0.30, random_state=42)
modelo_base = LinearRegression().fit(X_train_b, y_train_b)
r2_base = r2_score(y_test_b, modelo_base.predict(X_test_b))

# Modelo MEJORADO: agregamos grade, lat, long, waterfront, view
X_mejorado = df[["sqft_living", "bathrooms", "yr_built", "grade", "lat", "long", "waterfront", "view"]]

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_mejorado, y, test_size=0.30, random_state=42)
modelo_mejorado = LinearRegression().fit(X_train_m, y_train_m)

y_pred_m = modelo_mejorado.predict(X_test_m)
r2_mejorado = r2_score(y_test_m, y_pred_m)
rmse_mejorado = np.sqrt(mean_squared_error(y_test_m, y_pred_m))

print("*** Comparación de modelos ***")
print(f"R² modelo base (3 variables): {r2_base:.4f}")
print(f"R² modelo mejorado (8 variables): {r2_mejorado:.4f}")
print(f"RMSE modelo mejorado: {rmse_mejorado:.2f}")

print("\nCoeficientes del modelo mejorado:")
for var, coef in zip(X_mejorado.columns, modelo_mejorado.coef_):
    print(f"  {var}: {coef:.4f}")

*** Comparación de modelos ***
R² modelo base (3 variables): 0.5163
R² modelo mejorado (8 variables): 0.6801
RMSE modelo mejorado: 213511.28

Coeficientes del modelo mejorado:
  sqft_living: 158.7618
  bathrooms: 32982.1028
  yr_built: -2676.4485
  grade: 110416.4626
  lat: 547184.4832
  long: -65582.5378
  waterfront: 569566.7234
  view: 53805.2099


Al incorporar mas variables al modelo como grade, lat, long, waterfront y view pasa de tener 51.6% al 68% de la variabilidad del precio, es una mejora significativa que confirma la hipotesis planteada en la actividad anterior: la ubicación y la calidad de construcción son factores determinantes que el modelo base (solo tamaño, baños y antigüedad) no estaba capturando.

####Interpretación de los nuevos coeficientes

* lat (547,184.48): coeficiente muy grande y positivo y confirma que la latitud (posición norte-sur dentro del condado) tiene un efecto fuerte sobre el precio.
* long (-65,582.54): coeficiente negativo; sugiere que moverse hacia el oeste (valores de longitud menos negativos/más al oeste, hacia el agua) se asocia a mayor precio, consistente con el valor de las propiedades cercanas al Puget Sound.
* waterfront (569,566.72): es el coeficiente más alto de todos. Tiene sentido: una propiedad frente al mar (waterfront=1) suma en promedio más de medio millón de dólares al precio, manteniendo todo lo demás constante. Es uno de los hallazgos más fuertes e intuitivos del modelo mejorado.
* grade (110,416.46): cada punto adicional en la escala de calidad de construcción se asocia con un incremento de 110,000 en el precio un efecto considerable y coherente con la teoría.
* view (53,805.21): cada punto adicional en la calificación de la vista suma 53,805 al precio.
* yr_built (-2,676.45): se mantiene negativo y de magnitud similar a la del modelo base (-2,832.96), lo cual es un hallazgo importante: incluso controlando por ubicación, calidad y vista, la antigüedad sigue mostrando un efecto negativo leve, contradiciendo aún más la intuición inicial de "casa nueva = más cara". Esto refuerza la conclusión de que, en este mercado, las casas antiguas bien ubicadas mantienen su valor.
*sqft_living (158.76) y bathrooms (32,982.10): ambos coeficientes bajaron respecto al modelo base (antes 268.03 y 64,174.67 respectivamente). Esto es un efecto esperado y estadísticamente coherente: parte del efecto que antes se le atribuía al tamaño y a los baños en realidad estaba correlacionado con la ubicación y calidad, variables que ahora están controladas explícitamente en el modelo.

####Conclusión

La mejora de R² de 0.5163 a 0.6801 demuestra que el enfoque de mejora iterativa agregar variables relevantes fue efectivo, sin necesidad de replantear completamente la pregunta de investigación. El hallazgo más valioso es la confirmación cuantitativa de que la ubicación (lat, long, waterfront) y la calidad (grade, view) son factores con mayor peso individual que el tamaño o la antigüedad, lo cual es coherente con el conocimiento de dominio del mercado inmobiliario.

###Regularización (Ridge y Lasso)

Probaremos si la regularización mejora aún más el modelo o si detectamos señales de sobreajuste.

In [3]:
from sklearn.linear_model import Ridge, Lasso

# Usamos las mismas variables del modelo mejorado
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_mejorado, y, test_size=0.30, random_state=42
)

# Ridge (regularización L2)
for alpha in [0.1, 1.0, 10, 100]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_m, y_train_m)
    r2_ridge = r2_score(y_test_m, ridge.predict(X_test_m))
    print(f"Ridge alpha={alpha} -> R²: {r2_ridge:.4f}")

print()

# Lasso (regularización L1)
for alpha in [0.1, 1.0, 10, 100]:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_m, y_train_m)
    r2_lasso = r2_score(y_test_m, lasso.predict(X_test_m))
    print(f"Lasso alpha={alpha} -> R²: {r2_lasso:.4f}")

Ridge alpha=0.1 -> R²: 0.6801
Ridge alpha=1.0 -> R²: 0.6800
Ridge alpha=10 -> R²: 0.6791
Ridge alpha=100 -> R²: 0.6688

Lasso alpha=0.1 -> R²: 0.6801
Lasso alpha=1.0 -> R²: 0.6801
Lasso alpha=10 -> R²: 0.6801
Lasso alpha=100 -> R²: 0.6798


###Resultados de regularización

Se evaluó la aplicación de regularización Ridge y Lasso sobre el modelo mejorado de 8 variables, probando distintos valores de alpha (0.1, 1, 10, 100). Los resultados muestran que ni Ridge ni Lasso mejoran el R² del modelo (que se mantiene en 0.6801 para valores bajos de alpha), y que valores altos de regularización incluso deterioran levemente el desempeño (R²=0.6688 con Ridge alpha=100). Esto indica que el modelo actual no presenta señales de sobreajuste, dado el amplio número de observaciones de entrenamiento en relación con el número de variables utilizadas (proporción ~1,891:1). Por lo tanto, se concluye que la regularización no es necesaria en esta etapa del proyecto, y que la vía más efectiva para seguir mejorando el modelo es continuar con el feature engineering (más variables relevantes) en lugar de ajustar la penalización.

###Hiperparámetros en la regresión logística

Ahora optimicemos el modelo de clasificación con GridSearchCV, agregando también más variables (no solo sqft_living) para ver si mejora la exactitud de 73.37%.

In [4]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Crear variable binaria usando la mediana como umbral
mediana_precio = df["price"].median()
df["precio_alto"] = (df["price"] > mediana_precio).astype(int)

# Modelo logístico mejorado con más variables
X_log_mejorado = df[["sqft_living", "grade", "lat", "long", "waterfront", "view"]]
y_log = df["precio_alto"]

X_train_lm, X_test_lm, y_train_lm, y_test_lm = train_test_split(
    X_log_mejorado, y_log, test_size=0.30, random_state=42
)

param_grid = {"C": [0.01, 0.1, 1, 10, 100], "solver": ["lbfgs", "liblinear"]}
grid = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring="accuracy")
grid.fit(X_train_lm, y_train_lm)

print("Mejores parámetros:", grid.best_params_)
print("Mejor accuracy (CV):", grid.best_score_)

# Evaluar en test con los mejores parámetros
mejor_modelo = grid.best_estimator_
y_pred_lm = mejor_modelo.predict(X_test_lm)

print("Accuracy en test:", accuracy_score(y_test_lm, y_pred_lm))

Mejores parámetros: {'C': 0.1, 'solver': 'lbfgs'}
Mejor accuracy (CV): 0.7979245437612319
Accuracy en test: 0.8015114127082048


###Interpretación de los resultados

Mejora sustancial en accuracy: al incorporar grade, lat, long, waterfront y view junto con sqft_living, el modelo pasó de clasificar correctamente 73.4% a 80.2% de las viviendas en precio alto/bajo una mejora significativa que confirma, igual que en la regresión lineal, que la ubicación y la calidad de construcción son factores clave que el modelo univariado no estaba capturando.

Consistencia entre validación cruzada y test (79.79% vs. 80.15%): ambos valores son muy cercanos entre sí, lo cual es una buena señal de que el modelo generaliza bien y no está sobreajustado a los datos de entrenamiento.

Sobre el hiperparámetro óptimo (C=0.1): GridSearchCV encontró que un valor de C relativamente bajo (mayor regularización) es el óptimo, lo cual sugiere que, con 6 variables, aplicar una regularización moderada ayuda a evitar que el modelo se ajuste demasiado a peculiaridades del conjunto de entrenamiento, mejorando así su capacidad de generalización.

###Conclusión

Se aplicó búsqueda de hiperparámetros mediante GridSearchCV (validación cruzada de 5 particiones) sobre un modelo de regresión logística ampliado con variables de ubicación y calidad de construcción grade, lat, long, waterfront, view. El accuracy mejoró de 73.37% modelo base univariado a 80.15% modelo mejorado, evaluado en el conjunto de prueba, con un valor óptimo de C=0.1 y solver='lbfgs'. La cercanía entre el accuracy de validación cruzada (79.79%) y el de prueba (80.15%) confirma que el modelo generaliza adecuadamente. Este resultado refuerza la conclusión obtenida en la regresión lineal: las variables de ubicación y calidad de construcción son determinantes clave del valor de una vivienda en el condado de King, más allá de sus características físicas básicas.

#Construcción del dashboard con Dash

In [5]:
!pip install dash jupyter-dash plotly dash-bootstrap-components -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 58.6 MB/s eta 0:00:00


###Preparar los datos que alimentarán el dashboard

Antes de construir la interfaz, guardamos en variables simples los resultados clave que ya obtuvimos, para no tener que re-ejecutar todo el pipeline dentro del dashboard.

In [6]:
import pandas as pd
import numpy as np

# Dataset limpio (ya lo tienes)
df_dash = df.copy()

# Resumen de resultados de los modelos (hardcodeados con tus resultados reales)
resultados_modelos = {
    "Regresión lineal (base)": {"R2": 0.5163, "RMSE": 262532.66},
    "Regresión lineal (mejorado)": {"R2": 0.6801, "RMSE": 213511.28},
    "Regresión logística (base)": {"Accuracy": 0.7337},
    "Regresión logística (mejorado)": {"Accuracy": 0.8015},
}

# Contraste de hipótesis
resultado_hipotesis = {
    "t_stat": -16.19,
    "p_value": 1.48e-58,
    "media_antes_1980": 504422.73,
    "media_desde_1980": 587481.35
}

###Layout completo del dashboard (versión Colab, con JupyterDash)

In [7]:
pip install "dash[cloud]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 5.7 MB/s eta 0:00:00


In [22]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import dash_bootstrap_components as dbc

app = Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])

# Enable Jupyter proxy for inline display in Colab
app.enable_jupyter_proxy = True

app.layout = dbc.Container([

    # ---- Encabezado ----
    dbc.Row([
        dbc.Col(html.H1("Análisis del Mercado Inmobiliario — King County, USA"), width=12),
        dbc.Col(html.P("Dashboard de resultados: contraste de hipótesis, regresión lineal y regresión logística."), width=12),
    ], style={"marginBottom": "20px", "marginTop": "20px"}),

    # ---- Tarjetas de métricas clave ----
    dbc.Row([
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("R² Modelo Lineal (mejorado)"),
            html.H2(f"{resultados_modelos['Regresión lineal (mejorado)']['R2']:.4f}")
        ])), width=3),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("RMSE Modelo Lineal"),
            html.H2(f"${resultados_modelos['Regresión lineal (mejorado)']['RMSE']:,.0f}")
        ])), width=3),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("Accuracy Regresión Logística"),
            html.H2(f"{resultados_modelos['Regresión logística (mejorado)']['Accuracy']*100:.2f}%")
        ])), width=3),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("p-valor Contraste de Hipótesis"),
            html.H2("< 0.001", style={"color": "green"})
        ])), width=3),
    ], style={"marginBottom": "30px"}),

    # ---- Filtro interactivo ----
    dbc.Row([
        dbc.Col([
            html.Label("Filtrar por año de construcción:"),
            dcc.RangeSlider(
                id="filtro-anio",
                min=int(df_dash["yr_built"].min()),
                max=int(df_dash["yr_built"].max()),
                value=[int(df_dash["yr_built"].min()), int(df_dash["yr_built"].max())],
                marks={y: str(y) for y in range(1900, 2020, 20)},
                tooltip={"placement": "bottom"}
            ),
        ], width=12)
    ], style={"marginBottom": "30px"}),

    # ---- Gráficos principales ----
    dbc.Row([
        dbc.Col(dcc.Graph(id="grafico-dispersión"), width=6),
        dbc.Col(dcc.Graph(id="grafico-histograma"), width=6),
    ]),

    dbc.Row([
        dbc.Col(dcc.Graph(id="grafico-boxplot"), width=6),
        dbc.Col(dcc.Graph(id="grafico-comparacion-r2"), width=6),
    ]),

    # ---- Narrativa / conclusiones ----
    dbc.Row([
        dbc.Col([
            html.H3("Conclusiones principales"),
            html.Ul([
                html.Li("Existe una diferencia estadísticamente significativa en el precio entre viviendas antes y después de 1980 (p < 0.001)."),
                html.Li("El tamaño habitable (sqft_living) es el predictor individual más fuerte del precio."),
                html.Li("Incorporar ubicación (lat, long, waterfront) y calidad (grade, view) mejoró el R² de 0.52 a 0.68."),
                html.Li("El mismo enfoque mejoró el accuracy de clasificación de 73.4% a 80.2%."),
                html.Li("La regularización no fue necesaria en el modelo lineal, pero sí ayudó levemente en el modelo logístico."),
            ])
        ], width=12)
    ], style={"marginTop": "30px", "marginBottom": "40px"}),

], fluid=True)


# ---- Callbacks (interactividad) ----
@app.callback(
    Output("grafico-dispersión", "figure"),
    Output("grafico-histograma", "figure"),
    Output("grafico-boxplot", "figure"),
    Input("filtro-anio", "value")
)
def actualizar_graficos(rango_anio):
    dff = df_dash[(df_dash["yr_built"] >= rango_anio[0]) & (df_dash["yr_built"] <= rango_anio[1])]

    fig_disp = px.scatter(dff, x="sqft_living", y="price", opacity=0.3,
                           title="Tamaño habitable vs. Precio",
                           labels={"sqft_living": "Sqft Living", "price": "Precio (USD)"})

    fig_hist = px.histogram(dff, x="price", nbins=50,
                             title="Distribución de precios",
                             labels={"price": "Precio (USD)"})

    dff["grupo"] = np.where(dff["yr_built"] < 1980, "Antes de 1980", "Desde 1980")
    fig_box = px.box(dff, x="grupo", y="price",
                      title="Precio por periodo de construcción",
                      labels={"price": "Precio (USD)", "grupo": ""})

    return fig_disp, fig_hist, fig_box


# Gráfico estático de comparación de R² (no depende del filtro)
fig_r2 = px.bar(
    x=["Base (3 var.)", "Mejorado (8 var.)"],
    y=[0.5163, 0.6801],
    title="Mejora del R² — Regresión Lineal",
    labels={"x": "Modelo", "y": "R²"},
    text=[0.5163, 0.6801]
)
fig_r2.update_traces(texttemplate='%{text:.4f}', textposition='outside')

@app.callback(
    Output("grafico-comparacion-r2", "figure"),
    Input("filtro-anio", "value")  # trigger dummy para inicializar
)
def mostrar_comparacion(_):
    return fig_r2


# ---- Ejecutar en Colab ----
app.run(port=8050, debug=True)

from app import app
app.run(
jupyter_mode="external",
debug=False,
port=8050
)


<IPython.core.display.Javascript object>

Dash app running on:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

##Crear la estructura de carpetas en Colab

In [9]:
import os

os.makedirs("proyecto-king-county", exist_ok=True)
os.makedirs("proyecto-king-county/notebooks", exist_ok=True)
os.makedirs("proyecto-king-county/assets", exist_ok=True)

# Copiar el dataset limpio ya procesado (para que el dashboard no dependa de pasos previos)
df.to_csv("proyecto-king-county/kc_house_data_clean.csv", index=False)

##Crear app.py (versión standalone, sin jupyter_dash)

In [15]:
%%writefile proyecto-king-county/app.py

import dash
from dash import dcc, html, Input, Output
import dash_bootstrap_components as dbc
import plotly.express as px
import pandas as pd
import numpy as np

# ---- Cargar datos ----
df_dash = pd.read_csv("proyecto-king-county/kc_house_data_clean.csv")

# ---- Resultados de los modelos (obtenidos en el análisis previo) ----
resultados_modelos = {
    "Regresión lineal (base)": {"R2": 0.5163, "RMSE": 262532.66},
    "Regresión lineal (mejorado)": {"R2": 0.6801, "RMSE": 213511.28},
    "Regresión logística (base)": {"Accuracy": 0.7337},
    "Regresión logística (mejorado)": {"Accuracy": 0.8015},
}

resultado_hipotesis = {
    "t_stat": -16.19,
    "p_value": 1.48e-58,
    "media_antes_1980": 504422.73,
    "media_desde_1980": 587481.35
}

# ---- Inicializar la app ----
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])
server = app.server  # necesario para despliegue (Binder/Render/Heroku)

app.title = "King County Housing Dashboard"

# ---- Layout ----
app.layout = dbc.Container([

    dbc.Row([
        dbc.Col(html.H1("Análisis del Mercado Inmobiliario — King County, USA"), width=12),
        dbc.Col(html.P("Dashboard de resultados: contraste de hipótesis, regresión lineal y regresión logística."), width=12),
    ], style={"marginBottom": "20px", "marginTop": "20px"}),

    dbc.Row([
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("R² Modelo Lineal (mejorado)"),
            html.H2(f"{resultados_modelos['Regresión lineal (mejorado)']['R2']:.4f}")
        ])), width=3),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("RMSE Modelo Lineal"),
            html.H2(f"${resultados_modelos['Regresión lineal (mejorado)']['RMSE']:,.0f}")
        ])), width=3),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("Accuracy Regresión Logística"),
            html.H2(f"{resultados_modelos['Regresión logística (mejorado)']['Accuracy']*100:.2f}%")
        ])), width=3),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("p-valor Contraste de Hipótesis"),
            html.H2("< 0.001", style={"color": "green"})
        ])), width=3),
    ], style={"marginBottom": "30px"}),

    dbc.Row([
        dbc.Col([
            html.Label("Filtrar por año de construcción:"),
            dcc.RangeSlider(
                id="filtro-anio",
                min=int(df_dash["yr_built"].min()),
                max=int(df_dash["yr_built"].max()),
                value=[int(df_dash["yr_built"].min()), int(df_dash["yr_built"].max())],
                marks={y: str(y) for y in range(1900, 2020, 20)},
                tooltip={"placement": "bottom"}
            ),
        ], width=12)
    ], style={"marginBottom": "30px"}),

    dbc.Row([
        dbc.Col(dcc.Graph(id="grafico-dispersion"), width=6),
        dbc.Col(dcc.Graph(id="grafico-histograma"), width=6),
    ]),

    dbc.Row([
        dbc.Col(dcc.Graph(id="grafico-boxplot"), width=6),
        dbc.Col(dcc.Graph(id="grafico-comparacion-r2"), width=6),
    ]),

    dbc.Row([
        dbc.Col([
            html.H3("Conclusiones principales"),
            html.Ul([
                html.Li("Existe una diferencia estadísticamente significativa en el precio entre viviendas antes y después de 1980 (p < 0.001)."),
                html.Li("El tamaño habitable (sqft_living) es el predictor individual más fuerte del precio."),
                html.Li("Incorporar ubicación (lat, long, waterfront) y calidad (grade, view) mejoró el R² de 0.52 a 0.68."),
                html.Li("El mismo enfoque mejoró el accuracy de clasificación de 73.4% a 80.2%."),
                html.Li("La regularización no fue necesaria en el modelo lineal, pero sí ayudó levemente en el modelo logístico."),
            ])
        ], width=12)
    ], style={"marginTop": "30px", "marginBottom": "40px"}),

], fluid=True)


# ---- Callbacks ----
@app.callback(
    Output("grafico-dispersion", "figure"),
    Output("grafico-histograma", "figure"),
    Output("grafico-boxplot", "figure"),
    Input("filtro-anio", "value")
)
def actualizar_graficos(rango_anio):
    dff = df_dash[(df_dash["yr_built"] >= rango_anio[0]) & (df_dash["yr_built"] <= rango_anio[1])]

    fig_disp = px.scatter(dff, x="sqft_living", y="price", opacity=0.3,
                           title="Tamaño habitable vs. Precio",
                           labels={"sqft_living": "Sqft Living", "price": "Precio (USD)"})

    fig_hist = px.histogram(dff, x="price", nbins=50,
                             title="Distribución de precios",
                             labels={"price": "Precio (USD)"})

    dff = dff.copy()
    dff["grupo"] = np.where(dff["yr_built"] < 1980, "Antes de 1980", "Desde 1980")
    fig_box = px.box(dff, x="grupo", y="price",
                      title="Precio por periodo de construcción",
                      labels={"price": "Precio (USD)", "grupo": ""})

    return fig_disp, fig_hist, fig_box


fig_r2 = px.bar(
    x=["Base (3 var.)", "Mejorado (8 var.)"],
    y=[0.5163, 0.6801],
    title="Mejora del R² — Regresión Lineal",
    labels={"x": "Modelo", "y": "R²"},
    text=[0.5163, 0.6801]
)
fig_r2.update_traces(texttemplate='%{text:.4f}', textposition='outside')

@app.callback(
    Output("grafico-comparacion-r2", "figure"),
    Input("filtro-anio", "value")
)
def mostrar_comparacion(_):
    return fig_r2


# ---- Ejecutar servidor ----
if __name__ == "__main__":
    app.run(debug=True, host="0.0.0.0", port=8050)


Overwriting proyecto-king-county/app.py


##Crear requirements.txt

In [11]:
%%writefile proyecto-king-county/requirements.txt
dash==2.17.1
dash-bootstrap-components==1.6.0
plotly==5.22.0
pandas==2.2.2
numpy==1.26.4
gunicorn==22.0.0
jupyter-server-proxy==4.1.2

Writing proyecto-king-county/requirements.txt


In [17]:
# notebooks/ejecutar_dashboard.ipynb (una sola celda)

import sys
sys.path.append("proyecto-king-county")
from app import app

app.run(host="0.0.0.0", port=8051, debug=False)

<IPython.core.display.Javascript object>

## Crear README.md

In [21]:
%%writefile proyecto-king-county/README.md
# Dashboard: Análisis del Mercado Inmobiliario - King County, USA

Este proyecto presenta un análisis completo del dataset [House Sales in King County](https://www.kaggle.com/datasets/harlfoxem/housesalesprediction), incluyendo:

- Análisis exploratorio de datos
- Contraste de hipótesis (t-test)
- Regresión lineal múltiple
- Regresión logística
- Mejora iterativa de modelos (feature engineering, regularización, ajuste de hiperparámetros)
- Dashboard interactivo con Dash y Plotly


## Ejecutar en Binder

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/julianse23/king-county-housing-dashboard1/HEAD)

## Estructura del proyecto

- `app.py`: código del dashboard
- `kc_house_data_clean.csv`: dataset limpio
- `notebooks/analisis.ipynb`: notebook completo del análisis
- `requirements.txt`: dependencias del proyecto

Overwriting proyecto-king-county/README.md


##Copiar el notebook al proyecto

In [18]:
from google.colab import files
print("Sube aquí tu notebook .ipynb exportado desde Colab:")
uploaded = files.upload()

import shutil
for filename in uploaded.keys():
    shutil.move(filename, f"proyecto-king-county/notebooks/{filename}")

Sube aquí tu notebook .ipynb exportado desde Colab:


Saving Proyecto_Final_.ipynb to Proyecto_Final_.ipynb


In [20]:
!zip -r proyecto-king-county.zip proyecto-king-county/

from google.colab import files
files.download("proyecto-king-county.zip")

updating: proyecto-king-county/ (stored 0%)
updating: proyecto-king-county/app.py (deflated 63%)
updating: proyecto-king-county/__pycache__/ (stored 0%)
updating: proyecto-king-county/__pycache__/app.cpython-313.pyc (deflated 49%)
updating: proyecto-king-county/requirements.txt (deflated 20%)
updating: proyecto-king-county/assets/ (stored 0%)
updating: proyecto-king-county/notebooks/ (stored 0%)
updating: proyecto-king-county/notebooks/Proyecto_Final_.ipynb (deflated 78%)
updating: proyecto-king-county/README.md (deflated 40%)
updating: proyecto-king-county/kc_house_data_clean.csv (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Conclusión

El proyecto demuestra que, si bien es posible obtener conocimiento valioso a partir de análisis simples estadística descriptiva, regresión con una sola variable, la comprensión profunda de un fenómeno como el precio de la vivienda requiere un proceso iterativo: explorar, cuestionar hipótesis iniciales, incorporar nuevas variables, comparar técnicas y comunicar los resultados de forma efectiva. El modelo final, aunque mejorado significativamente (R²=0.68, accuracy=80%), aún deja margen de mejora sugerido en el propio informe mediante técnicas adicionales como modelos no lineales, mayor detalle geográfico por zipcode o variables externas proximidad a servicios, tendencias de mercado, lo cual reafirma que la ciencia de datos es un proceso de refinamiento continuo y no un resultado definitivo y cerrado

# Bibliografía
Plotly Technologies Inc. (2024). Dash documentation & user guide. Dash by Plotly. https://dash.plotly.com/

Plotly Technologies Inc. (2024). Plotly Python graphing library. Plotly. https://plotly.com/python/

Dash Bootstrap Components. (2024). Dash Bootstrap Components documentation. https://dash-bootstrap-components.opensource.faculty.ai/

scikit-learn. (2023, 29 de agosto). Ridge. Scikit-learn documentation. https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html

scikit-learn. (2023, 29 de agosto). Lasso. Scikit-learn documentation. https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html

scikit-learn. (2023, 29 de agosto). GridSearchCV. Scikit-learn documentation. https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

Project Jupyter, Binder Team. (2024). Binder documentation. mybinder.org. https://mybinder.readthedocs.io/en/latest/

GitHub, Inc. (2024). GitHub Docs. GitHub. https://docs.github.com/

Harlfoxem. (2016). House Sales in King County, USA [Conjunto de datos]. Kaggle. https://www.kaggle.com/datasets/harlfoxem/housesalesprediction

Pandas Development Team. (2024). Pandas documentation. Pandas. https://pandas.pydata.org/docs/

NumPy Developers. (2024). NumPy documentation. NumPy. https://numpy.org/doc/stable/

Google. (2024). Google Colaboratory documentation. Google Research. https://colab.research.google.com

VanderPlas, J. (2016). Python data science handbook: Essential tools for working with data. O'Reilly Media.

James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). An introduction to statistical learning: With applications in Python. Springer.